In [3]:
import pandas as pd
import os

In [5]:
file_paths

[]

In [4]:
# Automatically list all .xlsx files in the directory
file_paths = [file for file in os.listdir() if file.endswith('.xlsx')]

dfs = []

# Process each file
for file_path in file_paths:
    # Extract region and scenario from the file name
    file_name = os.path.basename(file_path)
    region = file_name.split('_')[1].split('.')[0]  # Extract region from file name (e.g., Alberta, NewBrunswick)
    scenario = file_name.split('_')[0]  # Extract the scenario from the file name (e.g., test)
    
    # Open the Excel file and get sheet names
    xls = pd.ExcelFile(file_path)
    sheet_names = xls.sheet_names

    for sheet in sheet_names:
        df = pd.read_excel(file_path, sheet_name=sheet)

        # Process variable and unit from the sheet name
        variable, unit = sheet.split(' (')
        variable = variable.strip()  # Strip spaces from variable
        unit = unit.strip().rstrip(')')  # Strip spaces from unit and remove closing parenthesis

        # Rename the first column to 'time'
        df.rename(columns={df.columns[0]: 'time'}, inplace=True)

        # Melt the dataframe to have 'time' as id_vars, variable names in 'variable', and values in 'value'
        df = pd.melt(df, id_vars=['time'], var_name='variable', value_name='value')

        # Add the unit and processed variable names
        df['unit'] = unit
        df['variable'] = variable + '|' + df['variable']

        # Add region and scenario columns
        df['region'] = region
        df['scenario'] = scenario
        df['model'] = 'MESSAGEix-Canada'

        # Append the dataframe to the list
        dfs.append(df)

# Concatenate all dataframes into a single dataframe
final_df = pd.concat(dfs, ignore_index=True)

final_df = final_df[final_df['time'] >= 2020]
final_df.to_csv('msgresults_new.csv', index=False)

ValueError: No objects to concatenate